# Exploración inicial

Objetivo: comprender dimensiones, tipos, faltantes, duplicados, distribuciones y problemas de calidad. Ejecutar solo después de construir tablas reales. No se muestran resultados ficticios.

In [ ]:
from pathlib import Path
import sys, os
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'config/config.example.json').exists()), None)
if project_root is None:
    raise RuntimeError('Abre este notebook desde notebooks/ dentro del repositorio.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
data_root = Path(os.getenv('STOCK_DATA_ROOT', str(project_root/'data')))
print('Raíz del proyecto:', project_root)


In [ ]:
import pandas as pd
from IPython.display import display
prices = pd.read_parquet(data_root/'interim/prices.parquet')
news = pd.read_parquet(data_root/'interim/news.parquet')
model = pd.read_parquet(data_root/'processed/model_dataset.parquet')
for name, frame in [('precios',prices),('noticias',news),('modelo',model)]:
    print(name, frame.shape)
    display(frame.head())


In [ ]:
for name, frame in [('precios',prices),('noticias',news),('modelo',model)]:
    print(name)
    display(pd.DataFrame({'tipo':frame.dtypes.astype(str), 'faltantes':frame.isna().sum(), 'faltantes_pct':100*frame.isna().mean()}))
print('Claves duplicadas:', {'precios':int(prices.duplicated(['ticker','date']).sum()), 'noticias':int(news.duplicated('article_id').sum()), 'modelo':int(model.duplicated(['ticker','date']).sum())})


In [ ]:
display(model[['adjusted_close','return_pct','news_count','sentiment_mean','neg_intensity']].describe())
print('Fechas:', model.date.min(), model.date.max())
print('Sesiones con noticias:', model.has_news.mean())
display(model.loc[model.return_pct.abs().nlargest(10).index, ['date','return_pct','news_count','neg_intensity']])


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2,2,figsize=(12,7))
axes[0,0].plot(model.date, model.adjusted_close); axes[0,0].set_title('Cierre ajustado')
axes[0,1].hist(model.return_pct, bins=35); axes[0,1].set_title('Rendimientos (%)')
axes[1,0].plot(model.date, model.news_count); axes[1,0].set_title('Noticias por cierre')
axes[1,1].hist(news.sentiment, bins=35); axes[1,1].set_title('Sentimiento por noticia')
plt.tight_layout(); plt.show()


In [ ]:
from src.processing.explore_data import explore
reports_root = data_root/'reports' if str(data_root).startswith('/Volumes/') else project_root/'reports'
summary = explore(data_root, reports_root)


## Observaciones iniciales (completar tras ejecutar)
- Dimensiones y cobertura observadas: pendiente.
- Faltantes estructurales: primera fila warmup, primer rezago y última etiqueta; distinguir de errores.
- Noticias sin recuperación: has_news=false; no equivale a neutralidad ni exhaustividad.
- Días extremos y posibles causas documentadas: pendiente, no asumir causalidad por coincidencia.
- Problemas de calidad y registros rechazados: consultar processing_summary.json; pendiente interpretar.

## Preguntas abiertas
¿El acceso cubre todo el histórico? ¿Cuántas sesiones tienen noticias? ¿Hay revisiones del proveedor? ¿Hay señal incremental fuera de muestra? ¿El tamaño justifica una herramienta distribuida?
